# MongoDB Atlas Auto-Embedding Experiment Setup

This notebook validates your local setup and Atlas connection before running auto-embedding experiments.

In [8]:
# If needed, run this once in the notebook:
%pip install -r requirements.txt


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, time, json, statistics
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
import certifi

load_dotenv()
MONGODB_ATLAS_URI = os.getenv("MONGODB_ATLAS_URI")
if not MONGODB_ATLAS_URI:
    raise ValueError("MONGODB_ATLAS_URI is not set. Add it to your .env file.")

DB_NAME     = "sample_airbnb"
COLL_NAME   = "listingsAndReviews"
INDEX_NAME  = "auto_embed_summary_v4large"

client = MongoClient(MONGODB_ATLAS_URI, tlsCAFile=certifi.where())
coll   = client[DB_NAME][COLL_NAME]
print("Docs in collection:", coll.count_documents({}))

Docs in collection: 5555


In [11]:
index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "autoEmbed",
                "modality": "text",
                "path": "summary",
                "model": "voyage-4-large",
                # Optional tuning:
                "quantization": "scalar",   # float | scalar (default) | binary | binaryNoRescore
                "numDimensions": 1024       # 2048 | 1024 (default) | 512 | 256
            },
            { "type": "filter", "path": "address.country" },
            { "type": "filter", "path": "bedrooms" }
        ]
    },
    name=INDEX_NAME,
    type="vectorSearch",
)

result = coll.create_search_index(model=index_model)
print("Created:", result)

Created: auto_embed_summary_v4large


In [12]:
import time
while True:
    idx = next(i for i in coll.list_search_indexes() if i["name"] == INDEX_NAME)
    print(idx["status"], idx.get("queryable"))
    if idx["status"] == "READY":
        break
    time.sleep(10)

PENDING False
PENDING False
BUILDING False
BUILDING False
BUILDING False
BUILDING False
READY True


In [13]:
query_text = "cozy apartment near amusement parks for a family"

pipeline_symmetric = [
    {
        "$vectorSearch": {
            "index": INDEX_NAME,
            "path": "summary",
            "query": { "text": query_text },
            # "model" omitted -> uses voyage-4-large
            "filter": {
                "address.country": "United States",
                "bedrooms": { "$gte": 3 }
            },
            "numCandidates": 100,
            "limit": 5
        }
    },
    { "$project": { "_id": 0, "name": 1, "summary": 1,
                    "score": { "$meta": "vectorSearchScore" } } }
]

t0 = time.perf_counter()
sym_results = list(coll.aggregate(pipeline_symmetric))
sym_ms = (time.perf_counter() - t0) * 1000
print(f"Symmetric (voyage-4-large q): {sym_ms:.0f} ms")
for r in sym_results: print(round(r["score"],4), "-", r["name"])

Symmetric (voyage-4-large q): 761 ms
0.5042 - Kolea Resort at Waikoloa Beach
0.504 - 3BR Gramercy Apt in heart of NYC
0.5039 - Spacious Comtemporary 3BR Apartment
0.5039 - Cozy & convenient 3 bedroom w/ gym *Crown Heights*
0.5038 - Spacious 3 BDR APT Brooklyn Sleeps 12!


### Asymmetric query — index voyage-4-large, query voyage-4-lite

In [14]:
pipeline_asymmetric = [
    {
        "$vectorSearch": {
            "index": INDEX_NAME,
            "path": "summary",
            "query": { "text": query_text },
            "model": "voyage-4-lite",     # ← asymmetric override
            "filter": {
                "address.country": "United States",
                "bedrooms": { "$gte": 3 }
            },
            "numCandidates": 100,
            "limit": 5
        }
    },
    { "$project": { "_id": 0, "name": 1, "summary": 1,
                    "score": { "$meta": "vectorSearchScore" } } }
]

t0 = time.perf_counter()
asy_results = list(coll.aggregate(pipeline_asymmetric))
asy_ms = (time.perf_counter() - t0) * 1000
print(f"Asymmetric (voyage-4-lite q): {asy_ms:.0f} ms")
for r in asy_results: print(round(r["score"],4), "-", r["name"])

Asymmetric (voyage-4-lite q): 338 ms
0.5047 - Kolea Resort at Waikoloa Beach
0.5041 - Sands Of Kahana 356 - 3 Bedroom, 2 Bathroom | Sleeps 8
0.504 - 3BR Gramercy Apt in heart of NYC
0.504 - Spacious 3 BDR APT Brooklyn Sleeps 12!
0.504 - Family friendly vacation home near sandy beaches


### LangChain AutoEmbeddings — same asymmetric pattern

In [41]:
from langchain_mongodb.vectorstores import MongoDBAtlasVectorSearch
from langchain_mongodb.embeddings import AutoEmbeddings

client = MongoClient(
    MONGODB_ATLAS_URI,
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=30000,
)

collection = client[DB_NAME][COLL_NAME]

# Vector store backed by the auto-embed index above
vector_store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=AutoEmbeddings(model="voyage-4-large"),  # matches index
    index_name=INDEX_NAME,
    text_key="summary",
    embedding_key=None, # required for AutoEmbeddings
    relevance_score_fn=None, # required for AutoEmbeddings
)

# (Optional) Insert new docs — server-side embedded with voyage-4-large
# vector_store.add_texts(["A bright loft 5 minutes from the boardwalk."])

# Asymmetric retrieval:
# Easiest path is to drop to the raw $vectorSearch stage with model override,
# because the high-level similarity_search() will use the index's model by default.
results = vector_store.similarity_search(
    query="cozy apartment near amusement parks for a family",
    k=5,
    pre_filter={
        "address.country": "United States",
        "bedrooms": {"$gte": 3}
    },
    # Pass a different compatible Voyage-4 model for query-time embedding:
    additional={"model": "voyage-4-lite"},
)
for d in results:
    print(d.metadata.get("name"), "::", d.page_content[:80], "…")

Kolea Resort at Waikoloa Beach :: Beautiful, first floor condo which opens up to stunning landscape.  Light, airy  …
3BR Gramercy Apt in heart of NYC :: Perfect for weekend w/ family & friends, apt w/ 1 queen & 2 full beds in the hea …
Spacious Comtemporary 3BR Apartment :: Newly furnished three bedroom apartment in the heart of Park Slope, has all the  …
Cozy & convenient 3 bedroom w/ gym *Crown Heights* :: Cozy private apartment conveniently located 1 block from the train, walking dist …
Spacious 3 BDR APT Brooklyn Sleeps 12! :: This spacious, clean, and completely furnished 3-bedroom apartment located in Fl …


### Side-by-side comparison — latency, cost, overlap

In [42]:
queries = [
    "cozy apartment near amusement parks for a family",
    "luxury beach villa with a pool",
    "budget studio close to public transport",
    "historic downtown loft for two",
    "ski-in ski-out mountain cabin",
    "pet-friendly home with a fenced yard",
    "quiet room for a digital nomad with fast wifi",
    "rustic farmhouse with garden",
]

PRICE_PER_M_TOKENS = {           # USD per 1M tokens
    "voyage-4-large": 0.12,
    "voyage-4":       0.06,
    "voyage-4-lite":  0.02,
}

def run(model_name):
    lats, scores, hits = [], [], []
    for q in queries:
        stage = {
            "$vectorSearch": {
                "index": INDEX_NAME,
                "path":  "summary",
                "query": {"text": q},
                "numCandidates": 100,
                "limit": 5,
            }
        }
        if model_name:
            stage["$vectorSearch"]["model"] = model_name
        t0 = time.perf_counter()
        res = list(coll.aggregate([stage,
            {"$project": {"_id": 1, "score": {"$meta": "vectorSearchScore"}}}]))
        lats.append((time.perf_counter() - t0) * 1000)
        scores.append([r["score"] for r in res])
        hits.append([r["_id"] for r in res])
    return lats, scores, hits

lat_L, sc_L, hits_L = run("voyage-4-large")   # symmetric
lat_l, sc_l, hits_l = run("voyage-4-lite")    # asymmetric

# Top-5 overlap (Jaccard) between large-vs-lite for the same query
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(1, len(a | b))

overlap = [jaccard(a, b) for a, b in zip(hits_L, hits_l)]

# Approx cost per query (avg English query ~12 tokens; use your own measure)
EST_TOKENS_PER_QUERY = 12
def cost_per_query(m):
    return EST_TOKENS_PER_QUERY * PRICE_PER_M_TOKENS[m] / 1_000_000

print(f"Avg latency  large={statistics.mean(lat_L):.0f} ms   lite={statistics.mean(lat_l):.0f} ms")
print(f"Top-5 overlap (large vs lite): {statistics.mean(overlap):.2f}")
print(f"Per-query cost  large={cost_per_query('voyage-4-large')*1e6:.4f} µ$  "
      f"lite={cost_per_query('voyage-4-lite')*1e6:.4f} µ$  "
      f"({PRICE_PER_M_TOKENS['voyage-4-large']/PRICE_PER_M_TOKENS['voyage-4-lite']:.0f}× cheaper)")

Avg latency  large=287 ms   lite=268 ms
Top-5 overlap (large vs lite): 0.52
Per-query cost  large=1.4400 µ$  lite=0.2400 µ$  (6× cheaper)


### Cleanup (optional)

In [ ]:
coll.drop_search_index(INDEX_NAME)